!cd samples/01-fundamentals/lab-02-model-providers/01-ollama-model/

https://ollama.com/download/linux

curl -fsSL https://ollama.com/install.sh | sh

ollama serve

해당 모델은 사용하지 마세요. 파일 관련 tool 지원이 안됩니다.
ollama pull llama3.2:latest

https://ollama.com/library/gemma4

ollama pull gemma4:e2b

# ollama 에서 사용 가능한 모델

https://ollama.com/search

Ollama 모델 설정하기

In [23]:
import os
import nest_asyncio
import requests
from strands.models.ollama import OllamaModel
from strands import Agent, tool

# Allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

# Ollama 모델 구성
ollama_model = OllamaModel(
    #model_id="llama3.2:latest",
    model_id="gemma4:e2b",
    host="http://localhost:11434",
    params={
        "max_tokens": 2048,
        "temperature": 0.7,
        "top_p": 0.9,
        "stream": True,
    },
)

In [9]:
# File Operation Tools
@tool
def file_read(file_path: str) -> str:
    """Read a file and return its content. Supports both text and PDF files.

    Args:
        file_path (str): Path to the file to read

    Returns:
        str: Content of the file

    Raises:
        FileNotFoundError: If the file doesn't exist
    """
    try:
        # Check if it's a PDF file
        if file_path.lower().endswith('.pdf'):
            import PyPDF2
            with open(file_path, "rb") as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
                return text if text.strip() else "Error: Could not extract text from PDF"
        else:
            # Regular text file
            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()
    except FileNotFoundError:
        return f"Error: File '{file_path}' not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def file_write(file_path: str, content: str) -> str:
    """Write content to a file.

    Args:
        file_path (str): The path to the file
        content (str): The content to write to the file

    Returns:
        str: A message indicating success or failure
    """
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)

        with open(file_path, "w") as file:
            file.write(content)
        return f"File '{file_path}' written successfully."
    except Exception as e:
        return f"Error writing to file: {str(e)}"


@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories in the specified path.

    Args:
        directory_path (str): Path to the directory to list

    Returns:
        str: A formatted string listing all files and directories
    """
    try:
        items = os.listdir(directory_path)
        files = []
        directories = []

        for item in items:
            full_path = os.path.join(directory_path, item)
            if os.path.isdir(full_path):
                directories.append(f"Folder: {item}/")
            else:
                files.append(f"File: {item}")

        result = f"Contents of {os.path.abspath(directory_path)}:\n"
        result += (
            "\nDirectories:\n" + "\n".join(sorted(directories))
            if directories
            else "\nNo directories found."
        )
        result += (
            "\n\nFiles:\n" + "\n".join(sorted(files)) if files else "\nNo files found."
        )

        return result
    except Exception as e:
        return f"Error listing directory: {str(e)}"

In [25]:
system_prompt="You are a helpful AI assistant."

# 도구를 가진 에이전트 생성
local_agent = Agent(
    system_prompt=system_prompt,
    model=ollama_model,
    tools=[file_read, file_write, list_directory],
)

에이전트 사용하기

In [26]:
response = local_agent("/home/sagemaker-user 디렉토리에 있는 파일들을 보여주세요.")
response


Tool #1: list_directory
`/home/sagemaker-user` 디렉토리에는 다음과 같은 파일들과 디렉토리가 있습니다:

**디렉토리:**
*   .agent/
*   .cache/
*   .claude/
*   .config/
*   .ipynb_checkpoints/
*   .ipython/
*   .jupyter/
*   .kiro/
*   .local/
*   .npm/
*   .ollama/
*   .sagemaker_sql_editor_api_cache/
*   .virtual_documents/
*   samples/
*   user-default-efs/

**파일:**
*   .bashrc
*   01-first-agent-kr.ipynb
*   02-model-provider-kr.ipynb
*   untitled.chat

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '`/home/sagemaker-user` 디렉토리에는 다음과 같은 파일들과 디렉토리가 있습니다:\n\n**디렉토리:**\n*   .agent/\n*   .cache/\n*   .claude/\n*   .config/\n*   .ipynb_checkpoints/\n*   .ipython/\n*   .jupyter/\n*   .kiro/\n*   .local/\n*   .npm/\n*   .ollama/\n*   .sagemaker_sql_editor_api_cache/\n*   .virtual_documents/\n*   samples/\n*   user-default-efs/\n\n**파일:**\n*   .bashrc\n*   01-first-agent-kr.ipynb\n*   02-model-provider-kr.ipynb\n*   untitled.chat'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'list_directory': ToolMetrics(tool={'toolUseId': 'list_directory', 'name': 'list_directory', 'input': {'directory_path': '/home/sagemaker-user'}}, call_count=1, success_count=1, error_count=0, total_time=0.0006091594696044922)}, cycle_durations=[21.456226110458374], traces=[<strands.telemetry.metrics.Trace object at 0x7fef84155c70>, <strands.telemetry.metrics.Trace object at 0x7fef861d17f0>], accumulated_usage={'inputTo

In [27]:
response = local_agent.tool.list_directory("/home/sagemaker-user")
response

{'toolUseId': 'tooluse_list_directory_960523287',
 'status': 'success',
 'content': [{'text': 'Contents of /home/sagemaker-user:\n\nDirectories:\nFolder: .agent/\nFolder: .cache/\nFolder: .claude/\nFolder: .config/\nFolder: .ipynb_checkpoints/\nFolder: .ipython/\nFolder: .jupyter/\nFolder: .kiro/\nFolder: .local/\nFolder: .npm/\nFolder: .ollama/\nFolder: .sagemaker_sql_editor_api_cache/\nFolder: .virtual_documents/\nFolder: samples/\nFolder: user-default-efs/\n\nFiles:\nFile: .bashrc\nFile: 01-first-agent-kr.ipynb\nFile: 02-model-provider-kr.ipynb\nFile: untitled.chat'}]}

In [28]:
response = local_agent("'This is a test file created by my Ollama agent.'라는 내용이 담긴 'sample.txt' 파일을 생성하세요.")
response


Tool #2: file_write
`sample.txt` 파일이 `/home/sagemaker-user` 디렉토리에 성공적으로 생성되었습니다. 파일 내용은 다음과 같습니다:

"This is a test file created by my Ollama agent."

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '`sample.txt` 파일이 `/home/sagemaker-user` 디렉토리에 성공적으로 생성되었습니다. 파일 내용은 다음과 같습니다:\n\n"This is a test file created by my Ollama agent."'}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'list_directory': ToolMetrics(tool={'toolUseId': 'list_directory', 'name': 'list_directory', 'input': {'directory_path': '/home/sagemaker-user'}}, call_count=1, success_count=1, error_count=0, total_time=0.0006091594696044922), 'file_write': ToolMetrics(tool={'toolUseId': 'file_write', 'name': 'file_write', 'input': {'content': 'This is a test file created by my Ollama agent.', 'file_path': '/home/sagemaker-user/sample.txt'}}, call_count=1, success_count=1, error_count=0, total_time=0.0008046627044677734)}, cycle_durations=[21.456226110458374, 9.713615655899048], traces=[<strands.telemetry.metrics.Trace object at 0x7fef84155c70>, <strands.telemetry.metrics.Trace object at 0x7fef861d17f0>, <strands.telemetry.metri

In [29]:
response = local_agent("Read the file '이테크시스템_kiro_260318.pdf' and summarize it in 5 bullet points.")


Tool #3: file_read
제공해주신 '이테크시스템_kiro_260318.pdf' 파일의 내용을 5가지 핵심 요점으로 요약해 드립니다.

*   **Kiro의 정의:** Kiro는 AWS가 개발한 AI 기반 Agentic 개발 환경으로, GUI 개발 환경인 Kiro IDE와 터미널 환경인 Kiro CLI 두 가지 인터페이스를 제공합니다.
*   **Kiro IDE 주요 기능:** 아이디어를 요구사항, 설계, 태스크로 자동 변환하는 기능(Steering), 프로젝트 규칙을 AI에 지속 적용하는 기능(Hooks), 그리고 MCP + 규칙 + 자동화를 통한 원클릭 패키지 기능을 제공합니다.
*   **Kiro CLI의 역할:** Kiro CLI를 통해 자연어로 AWS를 운영할 수 있으며, 역할별 전문 에이전트, AWS 서비스 직접 연동(MCP), 팀 규칙 자동 적용(Steering), 도구 실행 전후 자동화(Hooks) 등을 수행합니다.
*   **기존 방식 대비 이점:** 기존의 AWS CLI 방식(명령어 숙지, 로그 직접 분석, 수동 보안 점검)과 달리, Kiro CLI는 **자연어 이해**를 통해 여러 서비스를 한 번에 조회하고, AI가 원인 분석 및 해결안을 제시하여 운영 효율성을 극대화합니다.
*   **사용 전 필수 조건:** Kiro CLI를 사용하기 위해서는 Kiro CLI 로그인(Builder ID 등)과 AWS 자격 증명(aws configure를 통한 IAM 권한 설정)이라는 두 가지 인증 절차가 필요합니다.